<a href="https://colab.research.google.com/github/shubhamjena28/traffic-sign-recognition-road-safety/blob/main/train/vision_pipeline_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Traffic Sign Recognition & Road Safety Vision Pipeline
### Multi-Model Perception, Temporal LSTM Risk, CVAE Generation, ONNX Export & Pure Java ADAS Engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shubhamjena28/traffic-sign-recognition-road-safety/blob/main/train/vision_pipeline_colab.ipynb)

This comprehensive Google Colab notebook implements the complete end-to-end vision engineering roadmap:
1. **ResNet-18 Classifier** on GTSRB crops with data augmentation (10 epochs)
2. **GoogLeNet (Inception v1) Classifier** with Top-1 Accuracy, Confusion Matrix, and Per-Class Recall
3. **Faster R-CNN (ResNet-50 FPN)** Object Detector fine-tuning on road safety detection sets
4. **Semantic Segmentation (DeepLabV3)** on BDD100K road scenes reporting Mean IoU and Pixel Accuracy
5. **Convolution Visualizations**: 2D Kernels, Feature Maps, 3D Conv on 5-frame temporal stack, and Instance Segmentation
6. **Unified Master Benchmark Table**: mAP@0.5, mAP@0.5:0.95, IoU, Missed-Sign Rates, Parameters, and CPU Latency
7. **Sequence Tracking**: YOLO + ResNet-18 tracking over video sequences (box growth as distance proxy)
8. **Temporal LSTM Risk Model**: 10-frame sliding windows; comparing Step, Sigmoid, and Tanh activations
9. **Conditional VAE (CVAE)**: Generates rare sign classes, projects 2D latent space with t-SNE, and measures recall delta
10. **ONNX Export & Verification**: Exporting YOLOv8, ResNet-18, and LSTM to ONNX with numerical tolerance checks
11. **Pure Java Application**: Image loading via `javax.imageio`, letterboxing, pure Java NMS, box drawing, and CLI commands (`detect`, `benchmark`, `demo-neurons`)

## Step 1: Verify Hardware Acceleration (GPU)

In [ ]:
!nvidia-smi
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

## Step 2: Clone Repository & Install Dependencies

In [ ]:
import os
if not os.path.exists("traffic-sign-recognition-road-safety"):
    !git clone https://github.com/shubhamjena28/traffic-sign-recognition-road-safety.git

%cd traffic-sign-recognition-road-safety

!pip install --quiet ultralytics opencv-python matplotlib pandas scikit-learn torchvision torch

## Step 3: Verify & Prepare Datasets

In [ ]:
# Step 3: Dataset Setup & Verification
# Checks for dataset presence and automatically generates starter images if cloning fresh
from pathlib import Path
import os

img_dir = Path('data/images/train')
train_imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')) if img_dir.exists() else []

if len(train_imgs) == 0:
    print('[INFO] Images not found in data/images/train (datasets are git-ignored).')
    print('[INFO] Generating starter dataset with Stop, Speed, and Danger signs...')
    !python data/create_sample_dataset.py

train_count = len(list(Path('data/images/train').glob('*.jpg')))
val_count = len(list(Path('data/images/val').glob('*.jpg')))
test_count = len(list(Path('data/images/test').glob('*.jpg')))
print(f'[✓] Dataset Ready: {train_count} train, {val_count} val, {test_count} test images in data/images')


### (Optional) Use Your Custom Dataset from Google Drive
If you uploaded your local `data/images` folder (from `C:\Users\HP\...\data\images`) to your Google Drive, run this cell to mount Drive and copy your dataset into Colab:

In [ ]:
# UNCOMMENT TO MOUNT GOOGLE DRIVE IF USING PERSONAL DATASET:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p data/images data/labels
# !cp -r '/content/drive/MyDrive/traffic-sign-data/images/'* data/images/
# !cp -r '/content/drive/MyDrive/traffic-sign-data/labels/'* data/labels/
# print(f'Copied custom dataset. Current images: {len(list(Path("data/images/train").glob("*.jpg")))}')

## Step 4: Train ResNet-18 Classifier on GTSRB Crops (with Augmentation)
Fine-tunes ResNet-18 with ColorJitter, RandomRotation, and RandomAffine.

In [ ]:
!python train/train_resnet18.py --epochs 10 --batch-size 32 --lr 0.001 --device auto

## Step 5: Train GoogLeNet (Inception v1) Classifier
Trains GoogLeNet on the same crops. Generates Confusion Matrix and Per-Class Recall.

In [ ]:
!python train/train_googlenet.py --epochs 10 --batch-size 32 --lr 0.001 --device auto

## Step 6: Fine-Tune Faster R-CNN (ResNet-50 FPN) Detector
Fine-tunes torchvision two-stage Faster R-CNN on the road sign detection set.

In [ ]:
!python train/train_faster_rcnn.py --epochs 5 --batch-size 4 --lr 0.0005 --device auto

## Step 7: Train Semantic Segmentation Model on BDD100K Subset
Trains DeepLabV3 on road scenes, reporting Mean IoU and Pixel Accuracy.

In [ ]:
!python train/train_segmentation.py --epochs 5 --batch-size 8 --lr 0.001 --device auto

## Step 8: Convolution Visualizations & Advanced Demos
Generates:
1. **2D Filter Kernels** (ResNet-18 conv1 7x7 spatial filters)
2. **Feature Maps** across ResNet-18 layers (conv1, layer1..4)
3. **3D Convolution Demo** on a 5-frame video approach stack (nn.Conv3d)
4. **Instance Segmentation Demo** with polygonal masks and boxes

In [ ]:
!python train/visualize_conv.py

# Display generated visualizations
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

fig_names = [
    "checkpoints/resnet_conv1_kernels.png",
    "checkpoints/resnet_feature_maps.png",
    "checkpoints/3d_conv_temporal_stack.png",
    "checkpoints/instance_segmentation_demo.png"
]

for f in fig_names:
    if Path(f).exists():
        plt.figure(figsize=(10, 6))
        plt.imshow(Image.open(f))
        plt.title(f)
        plt.axis("off")
        plt.show()

## Step 9: Comprehensive Unified Master Benchmark Table
Outputs mAP@0.5, mAP@0.5:0.95, IoU, missed-sign rates, params, and CPU ms/frame in ONE table.

In [ ]:
!python train/benchmark_all.py

with open("docs/benchmark_table.md") as f:
    from IPython.display import Markdown, display
    display(Markdown(f.read()))

## Step 10: Multi-Frame Sequence Tracking (YOLO + ResNet-18)
Processes consecutive dashcam video frames. Per frame, records:
- Number of detected signs
- Class groups (STOP, SPEED, DANGER, PEDESTRIAN)
- Largest box area ($w \times h$)
- Box growth rate ($\Delta Area_t = Area_t - Area_{t-1}$), serving as a proxy for vehicle closing distance.

In [ ]:
!python python/sequence_tracker.py --num-synthetic 30

import matplotlib.pyplot as plt
from PIL import Image
plt.figure(figsize=(10, 7))
plt.imshow(Image.open('checkpoints/sequence_tracking_growth.png'))
plt.axis('off')
plt.show()

## Step 11: Train Temporal LSTM Risk Model (10-Frame Windows)
Trains an LSTM on sliding windows of 10 frames to predict road hazard risk levels (LOW, MEDIUM, HIGH).
Compares **Step**, **Sigmoid**, and **Tanh** activations on the exact same task.

In [ ]:
!python train/train_lstm_risk.py --epochs 12

import matplotlib.pyplot as plt
from PIL import Image
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
ax1.imshow(Image.open('checkpoints/lstm_risk_confusion_matrix.png'))
ax1.set_title('LSTM Risk Confusion Matrix')
ax1.axis('off')
ax2.imshow(Image.open('checkpoints/lstm_sequence_risk.png'))
ax2.set_title('Approaching Sign Sequence Risk')
ax2.axis('off')
plt.show()

## Step 12: Train Conditional VAE on Rare Classes & Latent t-SNE
Trains a CVAE on underrepresented GTSRB classes, generates synthetic samples, plots the latent space via t-SNE, and measures rare-class recall improvement.

In [ ]:
!python train/train_cvae.py --epochs 15

import matplotlib.pyplot as plt
from PIL import Image
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
ax1.imshow(Image.open('checkpoints/cvae_generated_samples.png'))
ax1.set_title('Generated Synthetic Rare Sign Samples')
ax1.axis('off')
ax2.imshow(Image.open('checkpoints/cvae_latent_tsne.png'))
ax2.set_title('CVAE Latent Space Manifold (t-SNE)')
ax2.axis('off')
plt.show()

## Step 13: Export Models to ONNX & Verify Numerical Equivalence
Exports YOLOv8 (`model.export(format='onnx', imgsz=640)`), ResNet-18 (`torch.onnx.export`), and LSTM Risk to ONNX.
Verifies that ONNX Runtime predictions match PyTorch outputs within $10^{-4}$ tolerance.

In [ ]:
!python train/export_all_onnx.py

## Step 14: Pure Java Application Execution
Compiles and runs the Java ADAS engine with zero OpenCV setup issues:
1. `demo-neurons`: McCulloch-Pitts logic gates (AND/OR/NOT/XOR) & Sigmoid neuron
2. `benchmark`: Preprocessing, Inference, NMS, and Box Rendering latency & FPS
3. `detect`: Processes a folder of video frames, applies letterboxing, NMS, safety alerts, and box drawing

In [ ]:
# Compile Java components
!mkdir -p bin/core bin/app
!javac -d bin/core java-core/src/main/java/com/traffic/core/*.java
!javac -cp bin/core -d bin/app java-app/src/main/java/com/traffic/app/*.java

# Run CLI Commands
!java -cp "bin/core:bin/app" com.traffic.app.Main demo-neurons
!java -cp "bin/core:bin/app" com.traffic.app.Main benchmark --iterations 20
!java -cp "bin/core:bin/app" com.traffic.app.Main detect --input data/sequences/frames --output data/output_frames

## Step 15: Download All Generated Models & Artifacts
Bundles all trained checkpoints, ONNX graphs, visual plots, and the master benchmark table into one zip archive for immediate download.

In [ ]:
!zip -rq traffic_vision_artifacts.zip checkpoints docs/benchmark_table.md data/output_frames
print('[SUCCESS] Bundled all models, training curves, visualizations, and benchmark table into traffic_vision_artifacts.zip')

from google.colab import files
files.download('traffic_vision_artifacts.zip')